To build and run GenMol you need an API key from NVIDIA. If you do not have one, the script 'check_genmol_api_key.sh' should help you have a valid, secure and personal API key saved at ~/ngc/ngc_api_key.genmol

In [ ]:
API_KEY= None  # papermill overrides this when run non-interactively

In [ ]:
import os
if not API_KEY:
    with open(os.path.expanduser("~/.ngc/ngc_api_key.genmol")) as f:
        API_KEY = f.read().strip()
os.environ["NGC_API_KEY"] = API_KEY
os.environ["LOCAL_NIM_CACHE"] = "/nesi/nobackup/uoa04517/cache"

In [ ]:
#Start GenMol API server
import subprocess, time

log_path = "server.log"
proc = subprocess.Popen(
    "apptainer run --env NVIDIA_VISIBLE_DEVICES=0 --env NGC_API_KEY=$NGC_API_KEY "
    "--bind $LOCAL_NIM_CACHE:/home/nvs/.cache --writable-tmpfs --nv "
    f"genmol.sif /usr/local/bin/start_server > {log_path} 2>&1",
    shell=True,
)

time.sleep(2)  # assume server.log exists after a sleep
f = open(log_path)

while True:
    line = f.readline()
    if line:
        if "Uvicorn running" in line:
            break
    elif proc.poll() is not None:
        raise RuntimeError(f"Server exited early (code {proc.returncode}, check {log_path})")
    else:
        time.sleep(1)

print("GenMol server ready.")

In [ ]:
import requests

invoke_url = "http://localhost:8000/generate"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

payload = {
    "smiles": "C124CN3C1.S3(=O)(=O)CC.C4C#N.[*{20-20}]",
    "num_molecules": 500,
    "temperature": 1.0,
    "noise": 1.0,
    "step_size": 1,
    "scoring": "QED",   # Quantitative Estimate of Drug-likeness filtering
    "unique": True
}

response = requests.post(invoke_url, headers=headers, json=payload)
response.raise_for_status()
    
result = response.json()
generated_molecules = result.get("molecules", [])
    
print(f"Successfully generated {len(generated_molecules)} molecules:\n")
for idx, mol in enumerate(generated_molecules):
    print(f"Molecule {idx + 1}: {mol}")